## MCO1 A Posteriori Analysis of String-Matching Algorithms

## Hypotheses
Null Hypothesis ( ): There is no significant three-way interaction effect between Algorithm Type, Query Length, and the Number of Queries on the completion time of the string-matching process.

Alternative Hypothesis ( ): There is a significant three-way interaction effect between Algorithm Type, Query Length, and the Number of Queries on completion time. Specifically, as the Number of Queries increases to multiple simultaneous queries and Query Length increases, the Aho-Corasick algorithm will demonstrate a significantly lower completion time growth rate compared to the Naive and KMP algorithms.
 

Null Hypothesis ( ): There is no significant three-way interaction effect between Algorithm Type, Query Length, and the Number of Queries on the total number of operations executed.

Alternative Hypothesis ( ): There is a significant three-way interaction effect between Algorithm Type, Query Length, and the Number of Queries on the total number of operations. Specifically, as the Number of Queries and Query Length increase, the Aho-Corasick algorithm will execute significantly fewer total operations compared to the multiplicative operation growth seen in the Naive and KMP algorithms.



# Algorithm Implementation

### Naive/Brute-Force Algorithm

In [ ]:
def naive_search(text, queries):
    """
    Executes the Naive/Brute-Force string-matching algorithm.
    Accepts a single string or a list of query strings.
    Returns the match indices and the total number of character comparisons.
    """
    # Standardize input to a list to handle both Single and Multiple IV conditions
    if isinstance(queries, str):
        queries = [queries]
        
    results = {query: [] for query in queries}
    total_comparisons = 0                       # Modification to calculate comparisons
    n = len(text)
    
    # Outer loop: Process each query sequentially
    for query in queries:
        m = len(query)
        if m == 0 or m > n:
            continue
            
        # Text scanning loop: Stop when remaining text is shorter than the query
        for i in range(n - m + 1):
            match = True
            
            # Character comparison loop
            for j in range(m):
                total_comparisons += 1          # Modification to calculate comparisons
                if text[i + j] != query[j]:
                    match = False
                    break  # Mismatch found, shift text pointer by 1
            
            if match:
                results[query].append(i)
                
    return results, total_comparisons



### Knuth-Morris-Pratt (KMP) Algorithm

In [ ]:
def kmp_search(text, queries):
    """
    Executes the Knuth-Morris-Pratt (KMP) string-matching algorithm.
    Accepts a single string or a list of query strings.
    Returns the match indices and the total number of character comparisons.
    """
    # Standardize input to a list to handle both Single and Multiple IV conditions
    if isinstance(queries, str):
        queries = [queries]
        
    results = {query: [] for query in queries}
    total_comparisons = 0
    n = len(text)
    
    # Outer loop: Process each query sequentially
    for query in queries:
        m = len(query)
        if m == 0 or m > n:
            continue
            
        # Step 1: Preprocessing Phase - Build the Longest Prefix Suffix (LPS) array
        lps = [0] * m
        length = 0
        i = 1
        
        while i < m:
            total_comparisons += 1
            if query[i] == query[length]:
                length += 1
                lps[i] = length
                i += 1
            else:
                if length != 0:
                    length = lps[length - 1] # Fall back in the LPS array
                else:
                    lps[i] = 0
                    i += 1
                    
        # Step 2: Execution Phase - Scan the text using the LPS array
        i = 0  # text index
        j = 0  # query index
        
        while i < n:
            total_comparisons += 1
            if query[j] == text[i]:
                i += 1
                j += 1
                
                if j == m:
                    # Match found, record index and use LPS to avoid backtracking
                    results[query].append(i - j)
                    j = lps[j - 1]
            else:
                if j != 0:
                    j = lps[j - 1] # Skip matching characters using the prefix table
                else:
                    i += 1
                    
    return results, total_comparisons



### Aho-Corasick Algorithm

In [ ]:
from collections import deque

class AhoCorasick:
    def __init__(self):
        # Initialize the trie with a root node (index 0)
        self.trie = [{'children': {}, 'fail': 0, 'outputs': []}]
        self.total_operations = 0

    def add_word(self, word):
        """Step 1: Insert a query string into the Trie."""
        current_node = 0
        for char in word:
            self.total_operations += 1
            if char not in self.trie[current_node]['children']:
                # Create a new node
                self.trie.append({'children': {}, 'fail': 0, 'outputs': []})
                self.trie[current_node]['children'][char] = len(self.trie) - 1
            current_node = self.trie[current_node]['children'][char]
        self.trie[current_node]['outputs'].append(word)

    def build_automaton(self):
        """Step 2: Build failure links using Breadth-First Search (BFS)."""
        queue = deque()
        
        # Initialize failure links for depth 1 nodes to root (0)
        for char, child_node in self.trie[0]['children'].items():
            self.trie[child_node]['fail'] = 0
            queue.append(child_node)
            self.total_operations += 1

        while queue:
            current_node = queue.popleft()
            
            for char, child_node in self.trie[current_node]['children'].items():
                queue.append(child_node)
                fail_node = self.trie[current_node]['fail']
                self.total_operations += 1

                # Follow failure links until a matching character or root is found
                while fail_node != 0 and char not in self.trie[fail_node]['children']:
                    self.total_operations += 1
                    fail_node = self.trie[fail_node]['fail']

                if char in self.trie[fail_node]['children']:
                    self.total_operations += 1
                    self.trie[child_node]['fail'] = self.trie[fail_node]['children'][char]
                else:
                    self.trie[child_node]['fail'] = 0

                # Merge outputs to catch overlapping subset matches 
                # (e.g., finding "he" when matching "she")
                self.trie[child_node]['outputs'].extend(
                    self.trie[self.trie[child_node]['fail']]['outputs']
                )

    def search(self, text, queries):
        """Step 3: Scan the text in a single pass."""
        for q in queries:
            self.add_word(q)
        self.build_automaton()

        results = {q: [] for q in queries}
        current_node = 0

        for i, char in enumerate(text):
            self.total_operations += 1
            
            # Follow failure links on mismatch
            while current_node != 0 and char not in self.trie[current_node]['children']:
                self.total_operations += 1
                current_node = self.trie[current_node]['fail']

            if char in self.trie[current_node]['children']:
                self.total_operations += 1
                current_node = self.trie[current_node]['children'][char]
            else:
                current_node = 0

            # Record any matches found at the current state
            for word in self.trie[current_node]['outputs']:
                # i is the end index of the match; calculate the start index
                results[word].append(i - len(word) + 1)

        return results, self.total_operations

def aho_corasick_search(text, queries):
    """
    Wrapper function to maintain the same API as naive_search and kmp_search.
    """
    if isinstance(queries, str):
        queries = [queries]
    
    ac = AhoCorasick()
    return ac.search(text, queries)


# Testing Section

In [ ]:
import time
import random
import string
import polars as pl
import re

# 1. Pre-load narrative text once to avoid disk I/O overhead during the 16,200 runs
with open("aliceinwonderland.txt", "r", encoding="utf-8") as f:
    full_raw_text = f.read()

start_marker = "CHAPTER I."
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK"
s_idx = full_raw_text.find(start_marker)
e_idx = full_raw_text.find(end_marker)
ALICE_STORY = full_raw_text[s_idx:e_idx] if (s_idx != -1 and e_idx != -1) else full_raw_text
ALICE_WORDS = re.findall(r'\b[A-Za-z]+\b', ALICE_STORY)

CHAR_POOL = string.ascii_letters + string.digits + " "

def generate_pure_string(min_len, max_len, text_type="random", source_text=None):
    """
    Generates target texts via slicing, but generates narrative queries 
    by selecting whole English words.
    """
    length = random.randint(min_len, max_len)
    
    if text_type == "random":
        return "".join(random.choices(CHAR_POOL, k=length))
    else:
        # If source_text is provided, we are generating query strings
        if source_text: 
            # Find all whole words in the story that fit the required length bucket
            valid_words = [w for w in ALICE_WORDS if min_len <= len(w) <= max_len]
            if valid_words:
                return random.choice(valid_words)
        
        # If no source_text, we are generating the main target text to search through.
        # We continue to slice a large block of text.
        pool = ALICE_STORY
        start_pos = random.randint(0, len(pool) - length)
        return pool[start_pos : start_pos + length]

# 2. Define the experimental grid
ALGORITHMS = {
    "Naive": naive_search,
    "KMP": kmp_search,
    "Aho-Corasick": aho_corasick_search,
}

TEXT_TYPES = ["random", "narrative"]

TEXT_LENGTH_SPECS = [
    ("small", 10, 100),
    ("medium", 100, 1000),
    ("large", 1000, 10000),
]

QUERY_LENGTH_SPECS = [
    ("small", 1, 5),
    ("medium", 5, 10),
    ("large", 10, 20),
]

# 1 = "single", 2 through 10 = "multiple"
QUERY_COUNTS = list(range(1, 11))

NUM_TRIALS = 30

# 3. Exhaustive Execution Loop
rows = []
QUERY_INDEX = -1

for text_type in TEXT_TYPES:
    # Pre-generate 30 target texts per text_length category
    text_samples_by_length = {
        text_len_label: [
            generate_pure_string(t_min, t_max, text_type=text_type)
            for _ in range(NUM_TRIALS)
        ]
        for text_len_label, t_min, t_max in TEXT_LENGTH_SPECS
    }

    for query_len_label, q_min, q_max in QUERY_LENGTH_SPECS:
        for q_count in QUERY_COUNTS:
            query_type_label = "single" if q_count == 1 else "multiple"
            
            # Generate 30 samples of query sets (each set has q_count strings)
            query_batches = [
                [
                    generate_pure_string(q_min, q_max, text_type=text_type, source_text=ALICE_STORY)
                    for _ in range(q_count)
                ]
                for _ in range(NUM_TRIALS)
            ]
            
            # Evaluate across all 3 text lengths
            for text_len_label, _, _ in TEXT_LENGTH_SPECS:
                # Outer loop: Run each algorithm in a contiguous block of 30 trials
                for algo_name, algo_func in ALGORITHMS.items():
                    QUERY_INDEX += 1
                    for trial in range(NUM_TRIALS):
                        target_text = text_samples_by_length[text_len_label][trial]
                        queries = query_batches[trial]
                        
                        start_time = time.perf_counter_ns()
                        results, ops_count = algo_func(target_text, queries)
                        end_time = time.perf_counter_ns()
                        
                        # Check if at least one query matched
                        has_match = 1 if any(len(indices) > 0 for indices in results.values()) else 0
                        
                        # Sanitize strings to prevent CSV row-breaking in Excel
                        safe_target_text = target_text.replace('\n', ' ').replace('\r', '')
                        safe_queries = str(queries).replace('\n', ' ').replace('\r', '')
                        
                        rows.append({
                            "query_index": QUERY_INDEX,
                            "trial_num": trial,
                            "algorithm": algo_name,
                            "text_type": text_type,
                            "text_length": text_len_label,
                            "query_length": query_len_label,
                            "query_type": query_type_label,
                            "query_count": q_count,
                            "success_flag": has_match,
                            "elapsed_time (ns)": end_time - start_time,
                            "operations_count": ops_count,
                            "test_string": safe_target_text,
                            "query_strings": safe_queries,
                        })

# 4. Compile into Polars DataFrame and Export
df = pl.DataFrame(rows)
print(f"Total benchmark rows compiled: {df.height:,} (Expected: 540 combinations * 30 = 16,200)")
print(f"Unique query_index batches: {df['query_index'].n_unique()} (0 to {QUERY_INDEX})")
print(df.head(6))

# include_bom=True so that excel reads it properly
df.write_csv("string_matching_benchmark_results_varied_counts.csv", include_bom=True)

### Results Section

In [ ]:
import pandas as pd
import statsmodels.formula.api as smf
import polars as pl

# ==========================================
# STEP 5: Descriptive Statistics
# ==========================================
# 1. Original comprehensive condition summary
summary_df = (
    df.group_by(["algorithm", "text_type", "text_length", "query_length", "query_type", "query_count"])
    .agg([
        pl.len().alias("n_trials"),
        pl.col("elapsed_time (ns)").mean().alias("mean_time_ns"),
        pl.col("elapsed_time (ns)").std().alias("sd_time_ns"),
        pl.col("operations_count").mean().alias("mean_ops"),
        pl.col("operations_count").std().alias("sd_ops"),
        pl.col("success_flag").mean().alias("match_rate"),
    ])
    .sort(["algorithm", "text_type", "text_length", "query_length", "query_count"])
)

summary_df.write_csv("descriptive_statistics_varied_counts.csv")

# ==========================================
# STEP 6: Linear Mixed Models (LMM) Preparation
# ==========================================
pdf = pd.DataFrame(
    df.rename({"elapsed_time (ns)": "elapsed_time_ns"}).to_dict(as_series=False)
)

pdf["algorithm"] = pd.Categorical(pdf["algorithm"], categories=["Naive", "KMP", "Aho-Corasick"])
pdf["query_length"] = pd.Categorical(pdf["query_length"], categories=["small", "medium", "large"])
pdf["query_type"] = pd.Categorical(pdf["query_type"], categories=["single", "multiple"])

pdf["sample_id"] = (
    pdf["text_type"].astype(str) + "_" +
    pdf["text_length"].astype(str) + "_" +
    pdf["query_length"].astype(str) + "_" +
    pdf["query_count"].astype(str) + "_" +
    pdf["trial_num"].astype(str)
)

# ==========================================
# STEP 7: Fit and Print Models
# ==========================================
lmm_time = smf.mixedlm(
    "elapsed_time_ns ~ C(algorithm) * C(query_length) * query_count",
    data=pdf,
    groups=pdf["sample_id"]
).fit()

print("\n=== LMM Results: Completion Time (ns) ===")
print(lmm_time.summary())

lmm_ops = smf.mixedlm(
    "operations_count ~ C(algorithm) * C(query_length) * query_count",
    data=pdf,
    groups=pdf["sample_id"]
).fit()

print("\n=== LMM Results: Total Operations Count ===")
print(lmm_ops.summary())

# ==========================================
# STEP 8: Export LMM Results to CSV
# ==========================================
time_summary_df = lmm_time.summary().tables[1]
time_summary_df.to_csv("lmm_completion_time_results.csv")

ops_summary_df = lmm_ops.summary().tables[1]
ops_summary_df.to_csv("lmm_operations_count_results.csv")

print("\n[SUCCESS] All CSV exports completed successfully.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# STEP 9: Visualize LMM Results (Time)
# ==========================================
# 1. Dynamically extract parameters and confidence intervals
params = lmm_time.params
conf = lmm_time.conf_int()

# Map the exact statsmodels output strings to readable labels
terms_mapping = {
    'C(algorithm)[T.Aho-Corasick]': 'Aho-Corasick Setup (Base)',
    'C(algorithm)[T.KMP]': 'KMP Setup (Base)',
    'query_count': 'Query Count (Naive Scaling)',
    'C(algorithm)[T.KMP]:query_count': 'KMP:Query Count (Scaling diff)',
    'C(algorithm)[T.Aho-Corasick]:query_count': 'Aho-Corasick:Query Count (Scaling diff)',
    'C(query_length)[T.medium]': 'Query Length [Medium]',
    'C(query_length)[T.large]': 'Query Length [Large]'
}

plot_data = {'Term': [], 'Coef': [], 'Lower': [], 'Upper': []}

for stat_term, label in terms_mapping.items():
    if stat_term in params:
        plot_data['Term'].append(label)
        plot_data['Coef'].append(params[stat_term])
        plot_data['Lower'].append(conf.loc[stat_term, 0])
        plot_data['Upper'].append(conf.loc[stat_term, 1])

df_plot = pd.DataFrame(plot_data)
df_plot['Error_Lower'] = df_plot['Coef'] - df_plot['Lower']
df_plot['Error_Upper'] = df_plot['Upper'] - df_plot['Coef']

# 2. Set up the figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 3. Forest Plot
y_pos = np.arange(len(df_plot))[::-1]
ax1.errorbar(
    df_plot['Coef'], y_pos, 
    xerr=[df_plot['Error_Lower'][::-1], df_plot['Error_Upper'][::-1]], 
    fmt='o', color='#2c3e50', capsize=4, elinewidth=2, markersize=8
)
ax1.axvline(x=0, color='#e74c3c', linestyle='--', linewidth=2)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(df_plot['Term'])
ax1.set_xlabel('Coefficient Estimate (Impact on Time ns)')
ax1.set_title('Forest Plot of Model Coefficients (95% CI)')
ax1.grid(True, axis='x', linestyle=':', alpha=0.6)

# 4. Interaction Plot (Scaling Efficiency)
counts = np.linspace(pdf["query_count"].min(), pdf["query_count"].max(), 100)

# Extract baseline intercept and scaling drivers safely
intercept = params['Intercept']
naive_scale = params['query_count']

# Calculate equations using the extracted variables
time_naive = intercept + (naive_scale * counts)
time_kmp = (intercept + params['C(algorithm)[T.KMP]']) + \
           ((naive_scale + params['C(algorithm)[T.KMP]:query_count']) * counts)
time_ac = (intercept + params['C(algorithm)[T.Aho-Corasick]']) + \
          ((naive_scale + params['C(algorithm)[T.Aho-Corasick]:query_count']) * counts)

ax2.plot(counts, time_naive, label='Naive (Baseline)', color='#e74c3c', linewidth=2)
ax2.plot(counts, time_kmp, label='KMP', color='#f39c12', linewidth=2)
ax2.plot(counts, time_ac, label='Aho-Corasick', color='#2980b9', linewidth=2)

ax2.set_xlabel('Query Count')
ax2.set_ylabel('Projected Execution Time ')
ax2.set_title('Scaling Efficiency (Interaction Effects)')
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.savefig("lmm_time_visualization.png", dpi=300, bbox_inches='tight')
print("[SUCCESS] LMM visualizations saved to lmm_time_visualization.png.")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# ==========================================
# STEP 10: Visualize LMM Results (Operations)
# ==========================================
# 1. Dynamically extract parameters and confidence intervals
params = lmm_ops.params
conf = lmm_ops.conf_int()

# Map the exact statsmodels output strings to readable labels
terms_mapping = {
    'C(algorithm)[T.Aho-Corasick]': 'Aho-Corasick Setup (Base)',
    'C(algorithm)[T.KMP]': 'KMP Setup (Base)',
    'query_count': 'Query Count (Naive Scaling)',
    'C(algorithm)[T.KMP]:query_count': 'KMP:Query Count (Scaling diff)',
    'C(algorithm)[T.Aho-Corasick]:query_count': 'Aho-Corasick:Query Count (Scaling diff)',
    'C(query_length)[T.medium]': 'Query Length [Medium]',
    'C(query_length)[T.large]': 'Query Length [Large]'
}

plot_data = {'Term': [], 'Coef': [], 'Lower': [], 'Upper': []}

for stat_term, label in terms_mapping.items():
    if stat_term in params:
        plot_data['Term'].append(label)
        plot_data['Coef'].append(params[stat_term])
        plot_data['Lower'].append(conf.loc[stat_term, 0])
        plot_data['Upper'].append(conf.loc[stat_term, 1])

df_plot = pd.DataFrame(plot_data)
df_plot['Error_Lower'] = df_plot['Coef'] - df_plot['Lower']
df_plot['Error_Upper'] = df_plot['Upper'] - df_plot['Coef']

# 2. Set up the figure
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# 3. Forest Plot
y_pos = np.arange(len(df_plot))[::-1]
ax1.errorbar(
    df_plot['Coef'], y_pos, 
    xerr=[df_plot['Error_Lower'][::-1], df_plot['Error_Upper'][::-1]], 
    fmt='o', color='#2c3e50', capsize=4, elinewidth=2, markersize=8
)
ax1.axvline(x=0, color='#e74c3c', linestyle='--', linewidth=2)
ax1.set_yticks(y_pos)
ax1.set_yticklabels(df_plot['Term'])
ax1.set_xlabel('Coefficient Estimate (Impact on Operations Count)')
ax1.set_title('Forest Plot of Model Coefficients (95% CI)')
ax1.grid(True, axis='x', linestyle=':', alpha=0.6)

# 4. Interaction Plot (Scaling Efficiency)
counts = np.linspace(pdf["query_count"].min(), pdf["query_count"].max(), 100)

# Extract baseline intercept and scaling drivers safely
intercept = params['Intercept']
naive_scale = params['query_count']

# Calculate equations using the extracted variables
ops_naive = intercept + (naive_scale * counts)
ops_kmp = (intercept + params['C(algorithm)[T.KMP]']) + \
           ((naive_scale + params['C(algorithm)[T.KMP]:query_count']) * counts)
ops_ac = (intercept + params['C(algorithm)[T.Aho-Corasick]']) + \
          ((naive_scale + params['C(algorithm)[T.Aho-Corasick]:query_count']) * counts)

# Modified: Added linestyle='--' to the KMP plot
ax2.plot(counts, ops_naive, label='Naive (Baseline)', color='#e74c3c', linewidth=2)
ax2.plot(counts, ops_kmp, label='KMP', color='#f39c12', linewidth=2, linestyle='--')
ax2.plot(counts, ops_ac, label='Aho-Corasick', color='#2980b9', linewidth=2)

ax2.set_xlabel('Query Count')
ax2.set_ylabel('Projected Operations Count')
ax2.set_title('Scaling Efficiency (Interaction Effects)')
ax2.legend()
ax2.grid(True, linestyle=':', alpha=0.6)

plt.tight_layout()
plt.savefig("lmm_ops_visualization.png", dpi=300, bbox_inches='tight')
print("[SUCCESS] LMM visualizations saved to lmm_ops_visualization.png.")